# Voltage Vandals — Smart Warehouse Robot System

**Course:** VIP (Vertically Integrated Projects)  
**Team:** Voltage Vandals  

---

This notebook is the primary technical documentation for the repository. It summarizes the course codebase, explains how components connect, and provides operational workflows for mapping, navigation, inventory scanning, and monitoring.

## 1. Executive summary

The project implements an **autonomous warehouse inventory system** using TurtleBot3 robots. Robots map warehouse environments with SLAM, navigate with **Nav2**, capture shelf images with a USB/Pi camera, detect **AprilTag 36h11** markers, and persist inventory state in **MongoDB**. Operators can monitor stock and send navigation goals through a **Taipy** web dashboard.

### Goals

- Inventory monitoring and scanning on warehouse shelves
- Automated restocking support via location-aware tag metadata
- Dynamic task allocation and real-time inventory updates
- Centralized monitoring and control through a web UI

### Hardware (documented stack)

| Component | Role |
|-----------|------|
| TurtleBot3 Waffle Pi | Mobile base |
| Raspberry Pi 4 | Onboard compute |
| Pi Camera / USB camera | Shelf imaging (`/dev/video0`) |
| RPLidar C1 | SLAM and Nav2 (`/dev/ttyUSB0`) |

### Software stack

| Layer | Technologies |
|-------|----------------|
| Robotics | ROS 2, Nav2, Cartographer, TurtleBot3, `nav2_simple_commander` |
| Vision | OpenCV, `cv_bridge`, `pupil_apriltags` (tag36h11) |
| Data | MongoDB (`pymongo`) |
| UI | Taipy GUI, TCP bridge on port 5050 |
| Tags | AprilTag 3 (Java generator + pre-built PNGs) |

## 2. System architecture

Data flows from the robot through ROS topics, vision subprocesses, and MongoDB into the operator dashboard.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    Operator / Warehouse UI (Taipy)                       │
│   other_files/Web_interface/receiver.py  +  sender.py (optional)        │
│        ↑ MongoDB poll          ↑ TCP :5050          ↑ Nav2 action        │
└────────┬─────────────────────────┬────────────────────────┬─────────────┘
         │                         │                        │
         ▼                         ▼                        ▼
┌─────────────────┐    ┌──────────────────────┐    ┌─────────────────────┐
│ MongoDB         │◄───│ ROS 2 Python nodes    │───►│ Nav2 + AMCL + map   │
│ warehouse_      │    │ Inventory_scan.py     │    │ wafflepi_parms.yaml │
│ inventory       │    │ capture.py            │    └──────────┬──────────┘
└─────────────────┘    └──────────┬───────────┘               │
                                    │ subprocess               ▼
                                    ▼                    LiDAR / odom / TF
                         detect_apriltag.py (tag36h11)
                                    ▲
                         /camera/image_raw  ←  camera_pub (bringup)
```

**Important:** Inventory state is stored in MongoDB, not on custom ROS messages. ROS is used for sensing, localization, and navigation only.

## 3. Repository structure

| Path | Status | Description |
|------|--------|-------------|
| `src/irobot/` | **Primary** | Production ROS 2 package: maps, Nav2 launch, integrated scan + navigation |
| `src/inventory/` | Legacy | Initial development; launch files still reference old package name `nav` |
| `other_files/taipy_modules/` | Prototypes | Early Taipy UI experiments (static or partial DB) |
| `other_files/Web_interface/` | Integrated UI | `receiver.py` (dashboard + Nav2), `sender.py` (metrics bridge) |
| `other_files/data_entry/` | Setup | MongoDB seed scripts for known tags and storage config |
| `other_files/subprocess/` | Vision CLI | AprilTag detector used by `irobot` nodes (`Detected Tag ID:` format) |
| `other_files/Modified_files/` | Bringup | `camera_pub.py`, `robot.py` (TB3 + LiDAR + camera) |
| `other_files/generate_apriltags/` | Tooling | Java/Ant pipeline to generate printable tag36h11 PNGs |
| `docs/` | Documentation | This notebook and docs index |

## 4. ROS 2 package: `irobot` (primary)

The `irobot` package is the reference implementation. It installs launch files, Nav2 parameters, occupancy maps, and RViz configs via `setup.py`.

### 4.1 Python nodes

| Node script | Purpose | Key I/O |
|-------------|---------|----------|
| `Inventory_scan.py` | **Main workflow:** patrol shelf waypoints, run capture at each stop, update MongoDB with AMCL pose and shelf label | In: `/camera/image_raw`, `/amcl_pose`; Out: MongoDB collections |
| `capture.py` | One-shot scan: single image → AprilTag subprocess → DB update (test location hardcoded) | In: `/camera/image_raw`; Out: MongoDB |
| `navigate_to_points.py` | Waypoint navigation only; capture integration left as TODO | Nav2 `BasicNavigator` |

#### `Inventory_scan.py` — integrated patrol

- Defines warehouse waypoints (labels **A3→A1**, transition **s**, **C1–C3**, **B3–B1**).
- At each waypoint, spins up an embedded `CaptureNode` that:
  - Subscribes to `/amcl_pose` (TRANSIENT_LOCAL) for real robot position.
  - Subscribes to `/camera/image_raw` (BEST_EFFORT).
  - Invokes `detect_apriltag.py` via subprocess.
  - Updates `tag_metadata`, `inventory_scans`, `inventory_status`, `storage_status`.
- Unknown tags are inserted with `category: "Unknown"`; known tags update on shelf or location change.

### 4.2 Launch files

| Launch file | Use case |
|-------------|----------|
| `cartographer_mapping.launch.py` | Build a map with Google Cartographer + RViz |
| `slam_navigation.launch.py` | Nav2 bringup with `maps/warehouse_map.yaml` and `config/wafflepi_parms.yaml` |

### 4.3 Configuration and maps

| Asset | Role |
|-------|------|
| `config/wafflepi_parms.yaml` | Full Nav2 stack for TurtleBot3 Waffle Pi (AMCL, planners, controllers, BT) |
| `config/cartographer.lua` | 2D SLAM: `base_scan`, odom, 3.5 m max range |
| `maps/warehouse_map.yaml` + `.pgm` | Default navigation map |
| `maps/dubai_house_basic.yaml` + `.pgm` | Alternate test map |

### 4.4 Package install note

`setup.py` currently has **empty `console_scripts`**. Nodes are run directly, for example:

```bash
python3 src/irobot/irobot/Inventory_scan.py
```

After `colcon build`, prefer registering entry points for `ros2 run irobot <node>`.

## 5. ROS 2 package: `inventory` (legacy)

Marked in `read_me.md` as initial development — **do not use for new work** unless fixing package naming.

| Issue | Detail |
|-------|--------|
| Package name mismatch | Launch files call `get_package_share_directory('nav')` but the package is `inventory` |
| Missing dependency | `package.xml` references `custom_files`, not present in repo |
| Subprocess output | `inventory/subprocess/detect_apriltag.py` prints `done`; `irobot` expects `Detected Tag ID:` from `other_files/subprocess/` |

Evolution path documented in code:

1. `inventory/capture.py` — vision proof-of-concept  
2. `inventory/c_navigate_to_pose.py` — Nav2 waypoint driving  
3. `irobot/capture.py` — MongoDB + metrics  
4. `irobot/Inventory_scan.py` — full patrol + AMCL + shelf labels  
5. `Web_interface/receiver.py` — operator UI with Nav2 goals

## 6. Supporting modules (`other_files/`)

### 6.1 Web interface

| File | Role |
|------|------|
| `receiver.py` | Taipy dashboard on port 5000; shelf buttons send `NavigateToPose`; polls `tag_metadata`; optional TCP listener for live metrics |
| `sender.py` | Polls MongoDB every 5 s; pushes merged `storage_status` + `inventory_status` over TCP to receiver |
| `working` | Older variant of `Inventory_scan.py` (simpler waypoint set) |

### 6.2 Taipy prototypes (`taipy_modules/`)

Incremental UI experiments: buttons, metrics, tables, multi-page menus. `interface_development1.py` and `interface_development2.py` connect partially to MongoDB; metrics are often still static.

### 6.3 MongoDB seed scripts (`data_entry/`)

| Script | Purpose |
|--------|---------|
| `datbase_entry(case1).py` | Seeds 9 known tags (IDs 0–8) and `storage_config.total_units=9` |
| `database_enctry(case2).py` | Seeds 5 known tags; includes tags documented as missing for test scenarios |

### 6.4 AprilTag pipeline

- **Detection:** `other_files/subprocess/detect_apriltag.py` — family `tag36h11`, OpenCV + `pupil_apriltags`.
- **Generation:** `generate_apriltags/` — Java/Ant tools; `generated_tags/size_1888px/` holds printable markers (IDs 0–30).

### 6.5 Robot bringup (`Modified_files/`)

- `camera_pub.py` — publishes `sensor_msgs/Image` on `camera/image_raw` (320×240, V4L2).
- `robot.py` — launch description: TurtleBot3 state publisher, RPLidar C1, `turtlebot3_ros`, camera node.

## 7. MongoDB data model

Database name: **`warehouse_inventory`** (default URI: `mongodb://localhost:27017/`)

| Collection | Typical content |
|------------|-----------------|
| `storage_config` | `{ total_units: 9 }` |
| `tag_metadata` | Per-tag: `tag_id`, `name`, `category`, `shelf`, `location`, `occupied`, `update_count`, `is_update` |
| `inventory_scans` | Per scan: `tag_id`, `scanned_on`, `location`, `image_path`, `known`, `shelf` |
| `inventory_status` | Aggregates: `missing_units`, `unknown_units`, `updated_units` |
| `storage_status` | `occupied_units`, `empty_units`, `total_scanned`, etc. |

Seed the database before first patrol run using scripts in `other_files/data_entry/`.

## 8. ROS topics and actions

| Topic / action | Used by | Notes |
|----------------|---------|-------|
| `/camera/image_raw` | `capture.py`, `Inventory_scan.py` | Requires `camera_pub` or equivalent |
| `/amcl_pose` | `Inventory_scan.py` | Robot pose for DB `location` field |
| `/odom` | Nav2 stack | From wafflepi params / robot driver |
| `navigate_to_pose` (action) | `receiver.py` | GUI-driven shelf navigation |
| `map` | `map_server` via launch | `warehouse_map.yaml` default |

## 9. Operational workflows

### A. Environment mapping

1. Bring up robot drivers (LiDAR, base, odometry) — see `other_files/Modified_files/robot.py`.
2. Start Cartographer:
   ```bash
   ros2 launch irobot cartographer_mapping.launch.py
   ```
3. Drive the robot through the warehouse; save the map to `src/irobot/maps/`.

### B. Autonomous navigation

1. ```bash
   ros2 launch irobot slam_navigation.launch.py
   ```
2. Set initial pose in RViz or via `setInitialPose` in Python nodes.
3. Verify AMCL localization before inventory scripts.

### C. Automated inventory scan (recommended)

1. Start MongoDB and run a seed script from `other_files/data_entry/`.
2. Ensure `camera_pub` and Nav2 are running.
3. ```bash
   python3 src/irobot/irobot/Inventory_scan.py
   ```
4. Robot visits shelf waypoints; each stop triggers tag detection and DB updates.

### D. Single-shot test capture

```bash
python3 src/irobot/irobot/capture.py
```

Uses a hardcoded test location in code — suitable for bench testing, not production pose.

### E. Operator dashboard

1. MongoDB running with current data.
2. Optional: `python3 other_files/Web_interface/sender.py` for live metrics.
3. `python3 other_files/Web_interface/receiver.py` — Taipy UI + Nav2 shelf buttons.
4. Open browser at `http://localhost:5000` (default Taipy port).

## 10. Dependencies and prerequisites

### ROS 2 (declared in `package.xml`)

`rclpy`, `nav2_bringup`, `nav2_msgs`, `std_msgs`, `sensor_msgs`, `geometry_msgs`, Cartographer, TurtleBot3 packages (inventory also lists `slam_toolbox`, `tf2_ros`, etc.)

### Python (implicit — install on robot/dev machine)

- `opencv-python`, `cv_bridge`
- `pymongo`
- `pupil_apriltags`
- `nav2_simple_commander`
- `pandas`, `taipy` (dashboard)
- `tf_transformations` (where used)

### External packages not vendored in this repo

- `camera_pub` ROS package (see `Modified_files/camera_pub.py`)
- `rplidar_ros`, Cartographer ROS wrappers
- TurtleBot3 and Nav2 system install per [ROBOTIS TurtleBot3](https://github.com/ROBOTIS-GIT/turtlebot3) documentation

## 11. Known gaps and maintenance notes

| Issue | Impact | Suggested fix |
|-------|--------|----------------|
| Empty `console_scripts` in both packages | No `ros2 run` for nodes | Register `Inventory_scan`, `capture` in `irobot/setup.py` |
| Hardcoded `/home/faheem/...` paths | Not portable across machines | Use env vars or `ament_index` share paths |
| Two `detect_apriltag.py` output formats | Wrong pairing breaks detection | Standardize on `other_files/subprocess` version for irobot |
| `navigate_to_points.py` capture TODO | Patrol without scanning | Integrate `run_capture_node` pattern from `Inventory_scan.py` |
| `inventory` launches use package `nav` | Launches fail | Rename share dir or fix launch files |
| Duplicate `map_server` in `slam_navigation.launch.py` | Possible Nav2 conflict | Review launch composition |
| Taipy metrics sometimes static | Dashboard not live | Wire `sender.py` → `receiver.py` consistently |
| `package.xml` TODO license/description | Packaging polish | Fill metadata before release |

## 12. Further reading

### In this repository

- Root overview: `README.md`
- AprilTag generation: `other_files/generate_apriltags/README.md`
- UI screenshots: `other_files/IROBOT_Images/`

### External references

- [TurtleBot3](https://github.com/ROBOTIS-GIT/turtlebot3)
- [TurtleBot3 simulations](https://github.com/ROBOTIS-GIT/turtlebot3_simulations)
- [OpenCV](https://github.com/opencv/opencv)
- [Picamera2](https://github.com/raspberrypi/picamera2)
- [Nav2 documentation](https://navigation.ros.org/)
- [AprilTag generation](https://github.com/AprilRobotics/apriltag-generation)

---

*Documentation generated from analysis of the VIP course codebase. For questions or updates, extend this notebook or add sibling notebooks under `docs/`.*

In [ ]:
# Optional: verify repository paths from this notebook's location
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
KEY_PATHS = [
    REPO_ROOT / "src/irobot/irobot/Inventory_scan.py",
    REPO_ROOT / "src/irobot/launch/slam_navigation.launch.py",
    REPO_ROOT / "other_files/Web_interface/receiver.py",
    REPO_ROOT / "other_files/data_entry/database_enctry(case2).py",
]

print(f"Repository root: {REPO_ROOT.resolve()}")
print()
for p in KEY_PATHS:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p.relative_to(REPO_ROOT)}")